Created by Rishal on 05th May 2025. This program contains the recently designed algorithm for an operator to decide which tesseract to Propose. 

In [7]:
from typing import List, Dict, Tuple, Set, Optional, Any
from dataclasses import dataclass
from enum import Enum

# The Algorithm


In [8]:
class LockType(Enum):
    NONE = 0
    PREVOTE = 1
    PRECOMMIT = 2


@dataclass
class Tesseract:
    participants: List[int]
    height: Dict[int, int]  # participant -> height
    stage: bool  # True if locked, False if committed
    view: int

    def is_locked(self) -> bool:
        return self.stage

    def __eq__(self, other):
        if not isinstance(other, Tesseract):
            return False
        return (self.participants == other.participants and
                self.height == other.height and
                self.stage == other.stage and
                self.view == other.view)
    
    def __hash__(self):
        # Create a hashable representation of the Tesseract
        heights_tuple = tuple((p, h) for p, h in sorted(self.height.items()))
        return hash((tuple(sorted(self.participants)), heights_tuple, self.stage, self.view))


@dataclass
class TesseractInfo:
    height: int
    lock_type: LockType
    view: int

In [15]:
def process_prepare_messages(prepare_quorum: List[Tesseract], P: List[int]) -> Optional[List[Tesseract]]:
    """
    Process a quorum of Prepare messages as the operator during the Propose stage 
    
    Args:
        prepare_quorum: A list of Tesseract objects representing Prepare messages
        P: The list of participants in the current interaction
        
    Returns:
        The Tesseract to be proposed, or None if no proposal should be made
    """
    # 1. preparedQC ← Aggregated quorum of Prepare messages
    # (We already have this as input parameter prepare_quorum)
    
    # 2. Initialize empty set tesseractSet
    tesseract_set: Set[Tesseract] = set()
    
    # 3. Identify unique tesseracts in preparedQC
    for tesseract in prepare_quorum:
        # 3.1 Add unique tesseracts to the set
        tesseract_set.add(tesseract)
    
    # 4. Check for unique locked tesseract
    locked_tesseracts = [ts for ts in tesseract_set if ts.is_locked()]
    if len(locked_tesseracts) == 1 and len(tesseract_set) == 1:
        unique_locked = locked_tesseracts[0]
        if set(unique_locked.participants) == set(P):
            # 4.1 If tesseract.participants = P then Propose tesseract
            return [unique_locked]
        else:
            # 4.2 Else Propose nil
            return None
    
    # 5-7. Check if any locked tesseracts exist
    is_locked_present = False
    for tesseract in tesseract_set:
        if tesseract.is_locked():
            is_locked_present = True
            break
    
    if not is_locked_present:
        # 7. Propose extending the highest committed tesseracts and latest state
        # Note: Since we don't have implementation details for this step, we'll mark it with a placeholder
        # In actual implementation, you'd need to define how to find and extend the highest committed tesseracts
        return find_and_extend_highest_committed_tesseracts(tesseract_set)
    
    # 8-9. Collect all unique participants
    participant_arr = []
    for tesseract in tesseract_set:
        for p in tesseract.participants:
            if p not in participant_arr:
                participant_arr.append(p)
    
    # 10-11. Find max heights/stages/views for each participant
    max_tesseract: Dict[int, TesseractInfo] = {}
    
    for ts in tesseract_set:
        for p in ts.participants:
            height = ts.height[p]
            lock_type = LockType.PREVOTE if ts.is_locked() else LockType.PRECOMMIT
            view = ts.view
            
            if p not in max_tesseract:
                max_tesseract[p] = TesseractInfo(height=height, lock_type=lock_type, view=view)
            else:
                current = max_tesseract[p]
                
                # Compare using tiebreak rules: height, then stage, then view
                if height > current.height:
                    max_tesseract[p] = TesseractInfo(height=height, lock_type=lock_type, view=view)
                elif height == current.height:
                    if lock_type.value > current.lock_type.value:
                        max_tesseract[p] = TesseractInfo(height=height, lock_type=lock_type, view=view)
                    elif lock_type == current.lock_type and view > current.view:
                        max_tesseract[p] = TesseractInfo(height=height, lock_type=lock_type, view=view)
    
    # 12. Filter tesseracts based on max heights
    tesseract_set_copy = tesseract_set.copy()  # Create a copy to avoid modifying during iteration
    
    for ts in tesseract_set_copy:
        if ts.is_locked():
            for p in ts.participants:
                if max_tesseract[p].lock_type == LockType.PRECOMMIT:
                    if ts.height[p] <= max_tesseract[p].height:
                        tesseract_set.remove(ts)
                        break
                elif max_tesseract[p].lock_type == LockType.PREVOTE:
                    if ts.height[p] < max_tesseract[p].height:
                        tesseract_set.remove(ts)
                        break
                    elif ts.height[p] == max_tesseract[p].height and ts.view < max_tesseract[p].view:
                        tesseract_set.remove(ts)
                        break
    print(f"Tesseracts after filtering: {tesseract_set}")

    # 13-14. Initialize counter and map to track locked tesseracts
    num_locked = 0
    locked_participants_map = {}  # Key: Participant, Value: List of locked tesseracts
    
    # 15. Count locked tesseracts and populate the map
    for tesseract in tesseract_set:
        if tesseract.is_locked():
            num_locked += 1
            
            # For each participant in this locked tesseract
            for p in tesseract.participants:
                if p not in locked_participants_map:
                    locked_participants_map[p] = [tesseract]
                else:
                    locked_participants_map[p].append(tesseract)
    
    # 16. If no locked tesseracts, extend highest committed
    if num_locked == 0:
        # Propose extending the highest committed tesseracts and latest state 
        # for some participant set P'' subset of P'
        return find_and_extend_highest_committed_tesseracts(tesseract_set)
    
    # 17-18. Find participants with multiple locked tesseracts
    participants_with_multiple_locked = []
    for p in locked_participants_map.keys():
        if len(locked_participants_map[p]) > 1:
            participants_with_multiple_locked.append(p)
    
    # 19. Remove tesseracts that have any participant with multiple locked tesseracts
    
    for tesseract in tesseract_set:
        if tesseract.is_locked():
            should_remove = False
            for p in participants_with_multiple_locked:
                if p in tesseract.participants:
                    should_remove = True
                    break
            
            if should_remove:
                tesseract_set.remove(tesseract)
    
    # 20-21. Propose each remaining tesseract with participant set as subset of the original one
    tesseracts_to_propose = []

    for tesseract in tesseract_set:
        if set(tesseract.participants).issubset(set(P)):
            tesseracts_to_propose.append(tesseract)
            
    # 22-23. Make the proposal
    if tesseracts_to_propose:
        # Return the set of remaining tesseracts
        return tesseracts_to_propose
    else:
        # No valid locked tesseracts remain after filtering
        return None
    
def find_and_extend_highest_committed_tesseracts(tesseract_set: Set[Tesseract]) -> Optional[Tesseract]:
    # This empty tesseract signifies that the implementation extends the highest committed and the latest state.
    # This is a placeholder code. 
    defaultTesseract = Tesseract(
        participants = [],
        height = {},
        stage = True,
        view = 0
    )
    return [defaultTesseract]

# Testing

In [17]:
# Some sample tesseracts for testing
ts1 = Tesseract(participants=[1, 2, 3], height={1: 10, 2: 9, 3: 11}, stage=True, view=5)  # Locked

# Test with a single locked tesseract
prepare_quorum1 = [ts1]
result1 = process_prepare_messages(prepare_quorum1, [1, 2, 3])
print(f"Expected: Single Locked Tesseract {ts1}")
print(f"Test result: {result1}")

Expected: Single Locked Tesseract Tesseract(participants=[1, 2, 3], height={1: 10, 2: 9, 3: 11}, stage=True, view=5)
Test result: [Tesseract(participants=[1, 2, 3], height={1: 10, 2: 9, 3: 11}, stage=True, view=5)]


In [18]:
# Test Case 1: Full lock - Outdated locks
print("Test Case 1: Full lock - Outdated locks")
P1 = [1,2]
ts1a = Tesseract(participants=[1, 2], height={1: 1, 2: 1}, stage=True, view=9)  # Locked
ts1b = Tesseract(participants=[2, 9], height={2: 2, 9: 0}, stage=True, view=9)  # Locked
prepare_quorum1 = [ts1a, ts1b]
result1 = process_prepare_messages(prepare_quorum1, P1)
print(f"Expected: None (locked after filtering does not match participant set)")
print(f"Result: {result1}\n")

Test Case 1: Full lock - Outdated locks
Tesseracts after filtering: {Tesseract(participants=[2, 9], height={2: 2, 9: 0}, stage=True, view=9)}
Expected: None (locked after filtering does not match participant set)
Result: None



In [19]:
# Test Case 2: Full lock - Outdated locks
print("Test Case 2: Full lock - Outdated locks")
P2 = [1,2,3,4]
ts2a = Tesseract(participants=[1, 2], height={1: 1, 2: 1}, stage=True, view=9)  # Locked
ts2b = Tesseract(participants=[2, 9], height={2: 2, 9: 0}, stage=True, view=9)  # Locked
ts2c = Tesseract(participants=[3, 4], height={3: 1, 4: 1}, stage=True, view=9)  # Locked
ts2d = Tesseract(participants=[4, 8], height={4: 2, 8: 0}, stage=True, view=9)  # Locked

prepare_quorum2 = [ts2a, ts2b, ts2c, ts2d]
result2 = process_prepare_messages(prepare_quorum2, P2)
print(f"Expected: None (ts2b and ts2d locked after filtering)")
print(f"Result: {result2}\n")

Test Case 2: Full lock - Outdated locks
Tesseracts after filtering: {Tesseract(participants=[2, 9], height={2: 2, 9: 0}, stage=True, view=9), Tesseract(participants=[4, 8], height={4: 2, 8: 0}, stage=True, view=9)}
Expected: None (ts2b and ts2d locked after filtering)
Result: None



In [20]:
# Test Case 3: Full lock - Valid locks
print("Test Case 3: Full lock - Valid locks")
P3 = [1,2]
ts3a = Tesseract(participants=[1, 2], height={1: 1, 2: 1}, stage=True, view=9)  # Locked
ts3b = Tesseract(participants=[1, 2], height={1: 1, 2: 1}, stage=True, view=9)  # Locked

prepare_quorum3 = [ts3a, ts3b]
result3 = process_prepare_messages(prepare_quorum3, P3)
print(f"Expected: Propose ts3a")
print(f"Result: {result3}\n")

Test Case 3: Full lock - Valid locks
Expected: Propose ts3a
Result: [Tesseract(participants=[1, 2], height={1: 1, 2: 1}, stage=True, view=9)]



In [21]:
# Test Case 4: Full lock - Valid locks
print("Test Case 4: Full lock - Valid locks")
P4 = [1,2,3,4]
ts4a = Tesseract(participants=[1, 2], height={1: 1, 2: 1}, stage=True, view=9)  # Locked
ts4b = Tesseract(participants=[1, 2], height={1: 1, 2: 1}, stage=True, view=9)  # Locked
ts4c = Tesseract(participants=[3, 4], height={3: 2, 4: 2}, stage=True, view=9)  # Locked
ts4d = Tesseract(participants=[3, 4], height={3: 2, 4: 2}, stage=True, view=9)  # Locked

prepare_quorum4 = [ts4a, ts4b, ts4c, ts4d]
result4 = process_prepare_messages(prepare_quorum4, P4)
print(f"Expected: None (two different locks)")
print(f"Result: {result4}\n")

Test Case 4: Full lock - Valid locks
Tesseracts after filtering: {Tesseract(participants=[1, 2], height={1: 1, 2: 1}, stage=True, view=9), Tesseract(participants=[3, 4], height={3: 2, 4: 2}, stage=True, view=9)}
Expected: None (two different locks)
Result: [Tesseract(participants=[1, 2], height={1: 1, 2: 1}, stage=True, view=9), Tesseract(participants=[3, 4], height={3: 2, 4: 2}, stage=True, view=9)]



In [22]:
# Test Case 5: Full lock - Outdated and valid locks
print("Test Case 5: Full lock - Outdated and valid locks")
P5 = [1,2,3,4]
ts5a = Tesseract(participants=[1, 2], height={1: 1, 2: 1}, stage=True, view=9)  # Locked
ts5b = Tesseract(participants=[1, 2], height={1: 1, 2: 1}, stage=True, view=9)  # Locked
ts5c = Tesseract(participants=[3, 4], height={3: 1, 4: 1}, stage=True, view=9)  # Locked
ts5d = Tesseract(participants=[4, 9], height={4: 2, 9: 0}, stage=True, view=9)  # Locked

prepare_quorum5 = [ts5a, ts5b, ts5c, ts5d]
result5 = process_prepare_messages(prepare_quorum5, P5)
print(f"Expected: None (two different locks after filtering)")
print(f"Result: {result5}\n")

Test Case 5: Full lock - Outdated and valid locks
Tesseracts after filtering: {Tesseract(participants=[1, 2], height={1: 1, 2: 1}, stage=True, view=9), Tesseract(participants=[4, 9], height={4: 2, 9: 0}, stage=True, view=9)}
Expected: None (two different locks after filtering)
Result: [Tesseract(participants=[1, 2], height={1: 1, 2: 1}, stage=True, view=9)]



In [32]:
# Test Case 6: Partial lock - Outdated locks
print("Test Case 6: Partial lock - Outdated locks")
P6 = [1,2]
ts6a = Tesseract(participants=[1, 2], height={1: 1, 2: 1}, stage=True, view=9)  # Locked
ts6b = Tesseract(participants=[2, 9], height={2: 2, 9: 0}, stage=False, view=9)  # Committed

prepare_quorum6 = [ts6a, ts6b]
result6 = process_prepare_messages(prepare_quorum6, P6)
print(f"Expected: Propose extending ts6b and state")
print(f"Result: {result6}\n")

Test Case 6: Partial lock - Outdated locks
Tesseracts after filtering: {Tesseract(participants=[2, 9], height={2: 2, 9: 0}, stage=False, view=9)}
Expected: Propose extending ts6b and state
Result: Tesseract(participants=[], height={}, stage=True, view=0)



***
***

# Old Test Cases


In [ ]:

# Test Case 3: Multiple locked tesseracts
print("Test Case 3: Multiple locked tesseracts")
ts3a = Tesseract(participants=[1, 2, 3], height={1: 10, 2: 9, 3: 11}, stage=True, view=5)  # Locked
ts3b = Tesseract(participants=[2, 3, 4], height={2: 12, 3: 11, 4: 10}, stage=True, view=6)  # Locked
prepare_quorum3 = [ts3a, ts3b]
result3 = process_prepare_messages(prepare_quorum3, [1, 2, 3, 4])
print(f"Expected: None (multiple locked tesseracts)")
print(f"Result: {result3}\n")

# Test Case 4: Only committed tesseracts
print("Test Case 4: Only committed tesseracts")
ts4a = Tesseract(participants=[1, 2, 3], height={1: 10, 2: 9, 3: 11}, stage=False, view=5)  # Committed
ts4b = Tesseract(participants=[2, 3, 4], height={2: 8, 3: 7, 4: 9}, stage=False, view=4)  # Committed
prepare_quorum4 = [ts4a, ts4b]
result4 = process_prepare_messages(prepare_quorum4, [1, 2, 3, 4])
print(f"Expected: Extended version of highest committed tesseract")
print(f"Result: {result4}\n")

# Test Case 5: Mixed tesseracts with height-based filtering
print("Test Case 5: Mixed tesseracts with height-based filtering")
ts5a = Tesseract(participants=[1, 2, 3], height={1: 10, 2: 9, 3: 11}, stage=True, view=5)  # Locked
ts5b = Tesseract(participants=[1, 2, 3], height={1: 12, 2: 11, 3: 13}, stage=True, view=6)  # Locked with higher heights
ts5c = Tesseract(participants=[2, 3, 4], height={2: 8, 3: 7, 4: 9}, stage=False, view=4)  # Committed
prepare_quorum5 = [ts5a, ts5b, ts5c]
result5 = process_prepare_messages(prepare_quorum5, [1, 2, 3])
print(f"Expected: The tesseract with higher heights (ts5b)")
print(f"Result: {result5}\n")

# Test Case 6: Same height, different views
print("Test Case 6: Same height, different views")
ts6a = Tesseract(participants=[1, 2, 3], height={1: 10, 2: 10, 3: 10}, stage=True, view=5)  # Locked
ts6b = Tesseract(participants=[1, 2, 3], height={1: 10, 2: 10, 3: 10}, stage=True, view=8)  # Locked with higher view
prepare_quorum6 = [ts6a, ts6b]
result6 = process_prepare_messages(prepare_quorum6, [1, 2, 3])
print(f"Expected: The tesseract with higher view (ts6b)")
print(f"Result: {result6}\n")

# Test Case 7: Empty tesseract set
print("Test Case 7: Empty tesseract set")
prepare_quorum7 = []
result7 = process_prepare_messages(prepare_quorum7, [1, 2, 3, 4])
print(f"Expected: None (empty input)")
print(f"Result: {result7}\n")

# Test Case 8: Complex scenario with filtering
print("Test Case 8: Complex scenario with filtering")
ts8a = Tesseract(participants=[1, 2, 3], height={1: 10, 2: 9, 3: 11}, stage=True, view=5)  # Locked
ts8b = Tesseract(participants=[1, 2, 3], height={1: 8, 2: 7, 3: 9}, stage=True, view=4)  # Locked with lower heights
ts8c = Tesseract(participants=[2, 3, 4], height={2: 12, 3: 13, 4: 11}, stage=True, view=6)  # Locked with higher heights
ts8d = Tesseract(participants=[1, 4, 5], height={1: 7, 4: 8, 5: 9}, stage=False, view=3)  # Committed
prepare_quorum8 = [ts8a, ts8b, ts8c, ts8d]
result8 = process_prepare_messages(prepare_quorum8, [1, 2, 3, 4, 5])
print(f"Expected: None (multiple locked tesseracts after filtering)")
print(f"Result: {result8}\n")

In [ ]:
print("=== Proposal Operator Algorithm Test Cases ===\n")

# Test Case 1: Single locked tesseract with matching participants
print("Test Case 1: Single locked tesseract with matching participants")
ts1 = Tesseract(participants=[1, 2, 3], height={1: 10, 2: 9, 3: 11}, stage=True, view=5)  # Locked
prepare_quorum1 = [ts1]
result1 = process_prepare_messages(prepare_quorum1, [1, 2, 3])
print(f"Expected: Return the tesseract (participants match)")
print(f"Result: {result1}\n")

# Test Case 2: Single locked tesseract with different participants
print("Test Case 2: Single locked tesseract with different participants")
ts2 = Tesseract(participants=[1, 2, 3], height={1: 10, 2: 9, 3: 11}, stage=True, view=5)  # Locked
prepare_quorum2 = [ts2]
result2 = process_prepare_messages(prepare_quorum2, [1, 2, 3, 4])
print(f"Expected: None (participants don't match)")
print(f"Result: {result2}\n")

# Test Case 3: Multiple locked tesseracts
print("Test Case 3: Multiple locked tesseracts")
ts3a = Tesseract(participants=[1, 2, 3], height={1: 10, 2: 9, 3: 11}, stage=True, view=5)  # Locked
ts3b = Tesseract(participants=[2, 3, 4], height={2: 12, 3: 11, 4: 10}, stage=True, view=6)  # Locked
prepare_quorum3 = [ts3a, ts3b]
result3 = process_prepare_messages(prepare_quorum3, [1, 2, 3, 4])
print(f"Expected: None (multiple locked tesseracts)")
print(f"Result: {result3}\n")

# Test Case 4: Only committed tesseracts
print("Test Case 4: Only committed tesseracts")
ts4a = Tesseract(participants=[1, 2, 3], height={1: 10, 2: 9, 3: 11}, stage=False, view=5)  # Committed
ts4b = Tesseract(participants=[2, 3, 4], height={2: 8, 3: 7, 4: 9}, stage=False, view=4)  # Committed
prepare_quorum4 = [ts4a, ts4b]
result4 = process_prepare_messages(prepare_quorum4, [1, 2, 3, 4])
print(f"Expected: Extended version of highest committed tesseract")
print(f"Result: {result4}\n")

# Test Case 5: Mixed tesseracts with height-based filtering
print("Test Case 5: Mixed tesseracts with height-based filtering")
ts5a = Tesseract(participants=[1, 2, 3], height={1: 10, 2: 9, 3: 11}, stage=True, view=5)  # Locked
ts5b = Tesseract(participants=[1, 2, 3], height={1: 12, 2: 11, 3: 13}, stage=True, view=6)  # Locked with higher heights
ts5c = Tesseract(participants=[2, 3, 4], height={2: 8, 3: 7, 4: 9}, stage=False, view=4)  # Committed
prepare_quorum5 = [ts5a, ts5b, ts5c]
result5 = process_prepare_messages(prepare_quorum5, [1, 2, 3])
print(f"Expected: The tesseract with higher heights (ts5b)")
print(f"Result: {result5}\n")

# Test Case 6: Same height, different views
print("Test Case 6: Same height, different views")
ts6a = Tesseract(participants=[1, 2, 3], height={1: 10, 2: 10, 3: 10}, stage=True, view=5)  # Locked
ts6b = Tesseract(participants=[1, 2, 3], height={1: 10, 2: 10, 3: 10}, stage=True, view=8)  # Locked with higher view
prepare_quorum6 = [ts6a, ts6b]
result6 = process_prepare_messages(prepare_quorum6, [1, 2, 3])
print(f"Expected: The tesseract with higher view (ts6b)")
print(f"Result: {result6}\n")

# Test Case 7: Empty tesseract set
print("Test Case 7: Empty tesseract set")
prepare_quorum7 = []
result7 = process_prepare_messages(prepare_quorum7, [1, 2, 3, 4])
print(f"Expected: None (empty input)")
print(f"Result: {result7}\n")

# Test Case 8: Complex scenario with filtering
print("Test Case 8: Complex scenario with filtering")
ts8a = Tesseract(participants=[1, 2, 3], height={1: 10, 2: 9, 3: 11}, stage=True, view=5)  # Locked
ts8b = Tesseract(participants=[1, 2, 3], height={1: 8, 2: 7, 3: 9}, stage=True, view=4)  # Locked with lower heights
ts8c = Tesseract(participants=[2, 3, 4], height={2: 12, 3: 13, 4: 11}, stage=True, view=6)  # Locked with higher heights
ts8d = Tesseract(participants=[1, 4, 5], height={1: 7, 4: 8, 5: 9}, stage=False, view=3)  # Committed
prepare_quorum8 = [ts8a, ts8b, ts8c, ts8d]
result8 = process_prepare_messages(prepare_quorum8, [1, 2, 3, 4, 5])
print(f"Expected: None (multiple locked tesseracts after filtering)")
print(f"Result: {result8}\n")

In [ ]:
# Some sample tesseracts for testing
ts1 = Tesseract(participants=[1, 2, 3], height={1: 10, 2: 9, 3: 11}, stage=True, view=5)  # Locked
ts2 = Tesseract(participants=[1, 2, 3, 4], height={1: 8, 2: 7, 3: 9, 4: 10}, stage=False, view=4)  # Committed
ts3 = Tesseract(participants=[2, 3, 4], height={2: 12, 3: 11, 4: 10}, stage=True, view=6)  # Locked

# Test with a single locked tesseract
prepare_quorum1 = [ts1]
result1 = process_prepare_messages(prepare_quorum1, [1, 2, 3])
print(f"Expected: Single Locked Tesseract {ts1}")
print(f"Test 1 result: {result1}")

# Test with multiple tesseracts including locked and committed
prepare_quorum2 = [ts1, ts2, ts3]
result2 = process_prepare_messages(prepare_quorum2, [1, 2, 3, 4])
print(f"Expected: Propose New Extending as both Locked ")
print(f"Test 2 result: {result2}")

# Trash

In [ ]:

    # 13-16. Check for locked tesseracts after filtering
    num_locked = 0
    locked_ts = None
    
    for tesseract in tesseract_set:
        if tesseract.is_locked():
            num_locked += 1
            locked_ts = tesseract
    
    if num_locked == 1:
        if sorted(locked_ts.participants) == sorted(P):
            return locked_ts
        else:
            return None
    elif num_locked > 1:
        return None
    
    # 17-19. Final check if no locked tesseracts remain
    is_locked_present = False
    for tesseract in tesseract_set:
        if tesseract.is_locked():
            is_locked_present = True
            break
    
    if not is_locked_present:
        # Propose extending the highest committed tesseracts
        return find_and_extend_highest_committed_tesseracts(tesseract_set)
    
    # Default case - should not normally reach here
    print("Default Case")
    return None



In [ ]:
def find_and_extend_highest_committed_tesseracts(tesseract_set: Set[Tesseract]) -> Optional[Tesseract]:
    """
    Find and extend the highest committed tesseracts to propose a new tesseract.
    
    Args:
        tesseract_set: Set of tesseracts to search through
        
    Returns:
        A new tesseract proposal based on the highest committed tesseracts
    """
    # In the implementation, we need to:
    # 1. Find the highest committed tesseracts (where stage=False)
    # 2. Determine the latest state
    # 3. Create a new tesseract that extends these
    
    committed_tesseracts = [ts for ts in tesseract_set if not ts.is_locked()]
    if not committed_tesseracts:
        return None
        
    # Sort by some criteria to find "highest" committed tesseracts
    # Need actual implementation
    highest_committed = max(committed_tesseracts, 
                           key=lambda ts: sum(ts.height.values()) * 1000 + ts.view)
    
    # Create a new tesseract that extends the highest committed one
    # Need actual implementation
    new_proposal = Tesseract(
        participants=highest_committed.participants.copy(),
        height={p: h + 1 for p, h in highest_committed.height.items()},
        stage=True,  # Proposing as locked
        view=highest_committed.view + 1
    )
    
    return new_proposal